<a href="https://colab.research.google.com/github/chin1230/cs4200/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PageRank

In this exercise, you will compute PageRank on a collection of 469,235 web sites using the iterative version of the PageRank algorithm described in class for sparse graphs.  As we discussed in class, this avoids performing cubic-time operations on a graph with a large number of nodes.

Furthermore, you should take advantage of the _sparsity_ of the original hyperlinks, even though "teleportation" or random jumps connect every note to every other node.  As long as the maximum degree of the original link graph is much less than the number of nodes, you should be able to keep each iteration's runtime less than quadratic in the number of nodes.

Consider the following directed graph:

![A directed link graph](https://www.khoury.northeastern.edu/home/dasmith/courses/cs6200//pagerank.jpg)

We can represent this graph as a collection of nodes, here, ordered pairs of node index and node name:

In [1]:
small_nodes = [(0, 'A'),
              (1, 'B'),
              (2, 'C'),
              (3, 'D'),
              (4, 'E'),
              (5, 'F')]

and a collection of directed links, i.e., ordered pairs from source to target:

In [2]:
small_edges = [
  (0, 1),
  (0, 2),
  (0, 5),
  (1, 2),
  (1, 3),
  (1, 4),
  (1, 5),
  (2, 3),
  (2, 4),
  (3, 0),
  (3, 2),
  (3, 4),
  (3, 5),
  (4, 0),
  (5, 0),
  (5, 1),
  (5, 4)
]

We use integer identifiers for the nodes for efficiency. In most real-world collections of hyperlinks, unlike this example, not every page will have in-links, nor will every page have out-links.

## First Implementation and Test

\[10 points\] Implement the iterative PageRank algorithm we discussed in class. Test your code on the six-node example using the input representation given above.  Be sure that your code handles pages that have no in-links or out-links properly.  (You may wish to test on a few such examples.) In later parts of this assignment, depending on how you store the data, it may be convenient to use iterators rather than storing the data in memory.

In [3]:
# TODO: Implement PageRank, given nodes and edges, to start with a uniform
# distribution over nodes, run a fixed number of iterations, and
# return a distribution over nodes.

def page_rank_fixed_iter(nodes, edges, iterations=10):
    nodes = list(nodes)
    edges = list(edges)
    n = len(nodes)
    damping = 0.85

    out_links = {node_id: [] for node_id, name in nodes}

    for source, target in edges:
        out_links[source].append(target)

    ranks = {node_id: 1 / n for node_id, name in nodes}

    for _ in range(iterations):
        dangling_sum = sum(
            ranks[node_id]
            for node_id, name in nodes
            if not out_links[node_id]
        )

        new_ranks = {
            node_id: (1 - damping) / n + damping * dangling_sum / n
            for node_id, name in nodes
        }

        for source, name in nodes:
            if out_links[source]:
                share = damping * ranks[source] / len(out_links[source])

                for target in out_links[source]:
                    new_ranks[target] += share

        ranks = new_ranks

    result = [
        (node_id, name, ranks[node_id])
        for node_id, name in nodes
    ]

    for node_id, name, rank in result:
        print(node_id, name, rank)

    return result

# Output PageRank on the toy graph at various points.
# Make sure your output has node number, name, and PageRank value.
page_rank_fixed_iter(small_nodes, small_edges, 1)
page_rank_fixed_iter(small_nodes, small_edges, 10)
page_rank_fixed_iter(small_nodes, small_edges, 100)

0 A 0.24930555555555556
1 B 0.11944444444444445
2 C 0.14305555555555555
3 D 0.13125
4 E 0.2138888888888889
5 F 0.14305555555555555
0 A 0.25203637602817186
1 B 0.13930650918251083
2 C 0.15129376593475075
3 D 0.11896297277689881
4 E 0.18710661014291716
5 F 0.15129376593475075
0 A 0.2521271053751956
1 B 0.1393061853185386
2 C 0.1513064898667053
3 D 0.11890782257353921
4 E 0.18704590699931614
5 F 0.1513064898667053


[(0, 'A', 0.2521271053751956),
 (1, 'B', 0.1393061853185386),
 (2, 'C', 0.1513064898667053),
 (3, 'D', 0.11890782257353921),
 (4, 'E', 0.18704590699931614),
 (5, 'F', 0.1513064898667053)]

## PageRank on Web Crawl Data

\[20 points\] Download and unpack a list of `.edu` websites and the links among them from the [Common Crawl](https://commoncrawl.org/2017/05/hostgraph-2017-feb-mar-apr-crawls/) open-source web crawl. For the sake of brevity, the data record links among websites, not web pages. The information for nodes and links is the same as the toy example above.

In [4]:
# If you're running on a machine (e.g., Windows) that doesn't have wget or gzip,
# feel free to comment this out and use a different set of commands to load
# the data.
!wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
!gzip -df vertices-edu.txt.gz
!wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
!gzip -df edges-edu.txt.gz

--2026-10-10 02:27:12--  https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
Resolving khoury.northeastern.edu (khoury.northeastern.edu)... 52.70.229.197
Connecting to khoury.northeastern.edu (khoury.northeastern.edu)|52.70.229.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3703486 (3.5M) [application/x-gzip]
Saving to: ‘vertices-edu.txt.gz’

vertices-edu.txt.gz 100%[===================>]   3.53M  3.03MB/s    in 1.2s    

2026-10-10 02:27:14 (3.03 MB/s) - ‘vertices-edu.txt.gz’ saved [3703486/3703486]

--2026-10-10 02:27:14--  https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
Resolving khoury.northeastern.edu (khoury.northeastern.edu)... 52.70.229.197
Connecting to khoury.northeastern.edu (khoury.northeastern.edu)|52.70.229.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12829526 (12M) [application/x-gzip]
Saving to: ‘edges-edu.txt.gz’

edges-edu.txt.gz    100%[===========

There should now be files `vertices-edu.txt` and `edges-edu.txt`.

In [5]:
# TODO: Process the raw data into the same format as the simple graph.
# You may create lists or iterators.

nodes = []
edges = []

with open('vertices-edu.txt', 'r') as f:
  for line in f:
    node_id, name = line.strip().split()
    nodes.append((int(node_id), name))

with open('edges-edu.txt', 'r') as f:
  for line in f:
    source, target = line.strip().split()
    edges.append((int(source), int(target)))


Refine your implementation of PageRank to test for numerical convergence. Specificially, at each iteration, calculate the [perplexity](https://en.wikipedia.org/wiki/Perplexity) of the PageRank distribution, where perplexity is defined as 2 raised to the [Shannon entropy](https://en.wikipedia.org/wiki/Entropy_(information_theory)) of the PageRank distribution, i.e., $2^{H(PR)}$. (Recall the definition of entropy from our discussion of data compression.) The maximum perplexity of a PageRank distribution will therefore be the number of nodes in the graph.

At each iteration, check the _change_ in perplexity. If the change is less than some threshold, you can stop.


In [6]:
# TODO: Implement convergence testing in PageRank
# If you choose, you can share some subroutines with your first version.
# Print the change in perplexity at each iteration.
import math

def page_rank(nodes, edges, threshold=1):
    nodes = list(nodes)
    n = len(nodes)
    damping = 0.85

    out_links = {node_id: [] for node_id, name in nodes}

    for source, target in edges:
        out_links[source].append(target)

    ranks = {node_id: 1 / n for node_id, name in nodes}

    previous_perplexity = n
    iteration = 0

    while True:
        iteration += 1

        new_ranks = {
            node_id: (1 - damping) / n
            for node_id, name in nodes
        }

        dangling_sum = 0

        for node_id, name in nodes:
            if len(out_links[node_id]) == 0:
                dangling_sum += ranks[node_id]

        for node_id, name in nodes:
            new_ranks[node_id] += damping * dangling_sum / n

        for source, name in nodes:
            if len(out_links[source]) > 0:
                share = damping * ranks[source] / len(out_links[source])

                for target in out_links[source]:
                    new_ranks[target] += share

        entropy = -sum(
            p * math.log2(p)
            for p in new_ranks.values()
            if p > 0
        )
        current_perplexity = 2 ** entropy

        change = abs(current_perplexity - previous_perplexity)

        print("Iteration:", iteration, "Perplexity change:", change)

        ranks = new_ranks

        if change < threshold:
            break

        previous_perplexity = current_perplexity

    return [
        (node_id, name, ranks[node_id])
        for node_id, name in nodes
    ]
# Run until perplexity changes by less than 1
PR = page_rank(nodes, edges, 1)

Iteration: 1 Perplexity change: 171163.55383266142
Iteration: 2 Perplexity change: 42328.63142879572
Iteration: 3 Perplexity change: 21490.54257957157
Iteration: 4 Perplexity change: 8276.316264756228
Iteration: 5 Perplexity change: 4436.956136796885
Iteration: 6 Perplexity change: 2005.0351724944776
Iteration: 7 Perplexity change: 1233.0420636029157
Iteration: 8 Perplexity change: 628.983496594592
Iteration: 9 Perplexity change: 433.92440696113044
Iteration: 10 Perplexity change: 238.63099273506668
Iteration: 11 Perplexity change: 184.15035957848886
Iteration: 12 Perplexity change: 106.70358255101019
Iteration: 13 Perplexity change: 89.46192193677416
Iteration: 14 Perplexity change: 54.54580442153383
Iteration: 15 Perplexity change: 48.293747459712904
Iteration: 16 Perplexity change: 30.87687932132394
Iteration: 17 Perplexity change: 28.293758322572103
Iteration: 18 Perplexity change: 18.867614385904744
Iteration: 19 Perplexity change: 17.593266451149248
Iteration: 20 Perplexity chang

## Link Analysis

\[20 points\] In this final section, you will compute some properties of this web-site graph.

First, consider the _in-link count_ of a website, simply the number of websites pointing to it (including self-links).

In [7]:
# TODO: List the document ID, domain name, and in-link count of the 70 websites with the highest in-link count

link_count = {node_id: 0 for node_id, name in nodes}

for source, target in edges:
    link_count[target] += 1

sorted_link_count = sorted(link_count.items(), key=lambda x: x[1], reverse=True)

top_70 = sorted_link_count[:70]

for node_id, count in top_70:
    for node in nodes:
        if node[0] == node_id:
            print(node[0], node[1], count)
            break

185524 edu.mit.web 4388
278032 edu.stanford 4021
244433 edu.purdue.english.owl 3531
140443 edu.indiana 3339
237176 edu.princeton 3251
64587 edu.columbia 3123
465503 edu.yale 2804
418623 edu.utexas 2622
383763 edu.unc 2592
197698 edu.nap 2494
439637 edu.washington 2291
373442 edu.umich 2281
440674 edu.washington.depts 2276
148945 edu.jhu.muse 2255
60975 edu.colorado 2232
449738 edu.wisc 2230
38320 edu.bu 2205
83572 edu.dartmouth 1965
408380 edu.usc 1952
178879 edu.mit 1946
27307 edu.berkeley 1908
233405 edu.pitt 1857
191069 edu.msu 1810
326371 edu.uchicago.press 1763
136464 edu.illinois 1753
93874 edu.educause 1741
56979 edu.cmu.cs 1730
199032 edu.ncsu 1709
36294 edu.brown 1702
202182 edu.nd 1689
68675 edu.cornell 1685
71095 edu.cornell.law 1646
183214 edu.mit.mitpress 1644
215627 edu.nyu 1625
56538 edu.cmu 1583
239378 edu.psu 1541
350412 edu.ufl 1533
120819 edu.harvard 1529
270369 edu.si 1513
107916 edu.gatech 1500
365396 edu.uky 1497
337138 edu.ucop 1482
358246 edu.uic 1472
382564 edu

Then, use the PageRank values compute by your second implementation. Note that some websites will have both a high in-link count and PageRank.

In [8]:
# TODO: List the document ID, domain name, and PageRank of the 70 websites with the highest PageRank.

sorted_PR = sorted(PR, key=lambda x: x[2], reverse=True)

top_70_PR = sorted_PR[:70]

for node_id, name, rank in top_70_PR:
    print(node_id, name, rank)

185524 edu.mit.web 0.0009280045215317963
465503 edu.yale 0.0007005900334388713
318278 edu.ucdavis.cas 0.0006415669573725832
278032 edu.stanford 0.000594466947342899
136464 edu.illinois 0.0005796990531325832
319795 edu.ucdavis.ice.code 0.000546531173461079
237176 edu.princeton 0.0005019558567343831
284517 edu.stanford.web 0.0004747836125750327
383763 edu.unc 0.000471677099661017
64587 edu.columbia 0.0004473680987421254
449738 edu.wisc 0.0004362073078530534
60975 edu.colorado 0.00043239569377725084
346969 edu.ucsf 0.00043004968390006896
140443 edu.indiana 0.0004296243789432212
42502 edu.byu.cas 0.00041783006127217625
334739 edu.uconn 0.00041278720125980214
14945 edu.arizona 0.00040927569589545786
244433 edu.purdue.english.owl 0.0004030289428298379
373442 edu.umich 0.00040207850914063194
439637 edu.washington 0.000398425238169731
408380 edu.usc 0.0003914546886961802
191069 edu.msu 0.0003913952221348782
281817 edu.stanford.med 0.0003913865284453883
418623 edu.utexas 0.0003819559079027653
2

Finally, compute some summary statistics on this dataset.

In [9]:
# TODO: Compute:
# - the proportion of websites with no in-links (i.e., source nodes);
n = len(nodes)
no_in_links = sum(count == 0 for count in link_count.values()) / n
# - the proportion of websites with no out-links (i.e., sink nodes);
out_count = {node_id: 0 for node_id, name in nodes}

for source, target in edges:
    out_count[source] += 1

no_out_links = sum(count == 0 for count in out_count.values()) / n
# - the proportion of websites whose PageRank is higher than the initial uniform distribution.
higher_PR = sum(rank > 1 / n for node_id, name, rank in PR) / n

print("Proportion with no in-links:", no_in_links)
print("Proportion with no out-links:", no_out_links)
print("Proportion with PageRank above initial value:", higher_PR)

Proportion with no in-links: 0.26153633041013563
Proportion with no out-links: 0.6106641661427643
Proportion with PageRank above initial value: 0.15514614212494807
